# Registry tutorial
This tutorial will show you how to:
- understand `Registry` objects and their key role in DNAStream. 
- add entities to a registry. 
- using Python special methods on a `Registry` object
- look up items in the registry by `label` or `id`.  
- update metadata of registered entities. 
- activate and deactivate registered entities. 
- query/export the registry or a subset of the registry to a `pandas.Dataframe`. 
- use `io` functions to simplify insertation from `maf` and `csv` files. 

In [1]:
from pathlib import Path
import tempfile
from dnastream import DNAStream

tmpdir = Path(tempfile.mkdtemp(prefix="dnastream_tutorial_"))
myfile = tmpdir / "temp.h5"

# Create the DNAStream file (fails if it already exists)
ds = DNAStream.create(path=str(myfile), patient_id="patientX", verbose=True)
ds.close()

print("Created:", myfile)

2026-02-05 13:48:56 - INFO - Created DNAStream file /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_nxm7obyj/temp.h5
2026-02-05 13:48:56 - INFO - Connection to DNAStream file '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_nxm7obyj/temp.h5' closed.


Created: /var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/dnastream_tutorial_nxm7obyj/temp.h5


## `Registry` objects and their role in DNAStream
Registries play a key role in DNAStream because they provide tracking and management of key research entities, such as samples, cells, SNVs, indels, SNPs. When multiple analysts are working on a project, it can be difficult to track which samples pass QA and are to be used indownstream analysis or what SNVs are the current active set when multiple SNV callers are run. It can be hard to manage metadata and easily format it to streamline bash and workflow scripts/pipelines. They also provide link to various measured data and computed analysis results. 

Formally, a registry is a structured 1-d append only dataset (think dataframe) stored in the DNAStream HDF5 file for perpetuity. Any modifications to the registry dataset are logged so registry event histories can be extracted. 

When streaming a DNAStream file, `Registry` objects serve as interfaces to the registry dataset. Via `Registry` objects, one can:   
- `add` new entities for registration
- query the registry to `get` a subset meeting selector critera, 
- `update` the metadata of editable fields
- `activate` entities by `id` or `label`
- `find` all entities by a given `label`
- extract all or part of the registry dataset `to_dataframe`


### The registry spine
Every registry dataset automatically includes the following eight fields:
1. id : this is a unique identifier (UUIDv4) string for every registered entity.
2. label : this an internally generated **non-unique** human-readable string used to identify entities. The schema will specify how this field is generated. 
3. idx : this is a internal index of the entity in the dataset
4. active : this is a boolean indicating if the entity is active. Registry policy is to allow at most one unique label to be active. 
5. created_at : timestamp (ISO8601 Z) the entity was registered
6. created_by : user that registered the entity
7. modified_at : timestamp (ISO8601 Z) the entity was last modified
8. modified_by : the user that last modified the entity.

*NOTE: Registry policy does not allow modification of registry spine fields.  It also prevents updated any fields used to generate the label field.*


### Accessing `Registry` objects
`DNAStream` currently includes three built-in registries. 
- `sample`
- `variant`
- `snp`

See the built-in schema documentation for more details on the fields these included. 

They can be accessed as attributes from a `DNAStream` object or by name via the `registry(name)` function within DNAStream.

In [2]:
with DNAStream.open(myfile, mode="r") as ds:
    #attribute access
    sample_reg = ds.sample
    variant_reg = ds.variant
    print(ds.snp)

    #registry acess function
    sample_reg = ds.registry("sample")
    print(sample_reg)
    print(ds.sample)


/registry/snp (Shape: (0,))
/registry/sample (Shape: (0,))
/registry/sample (Shape: (0,))


In [ ]:
from dnastream._builtin_schemas import REGISTRY_SCHEMAS
# View the schema of the sample registry
print(REGISTRY_SCHEMAS["sample"].to_markdown_table())



| name | dtype | required | validator |
| --- | --- | --- | --- |
| id | object | True | None |
| label | object | True | None |
| idx | <class 'numpy.int64'> | True | None |
| active | <class 'numpy.bool'> | True | None |
| created_at | object | True | None |
| created_by | object | True | None |
| modified_at | object | True | None |
| modified_by | object | True | None |
| sample_name | object | True | str_validator |
| organism | object | False | str_validator |
| library_strategy | object | False | str_validator |
| library_source | object | False | str_validator |
| library_selection | object | False | str_validator |
| library_layout | S10 | False | None |
| platform | object | False | str_validator |
| model | object | False | str_validator |
| center_name | object | False | str_validator |
| run | object | False | str_validator |
| study | object | False | str_validator |
| coverage | f4 | False | None |
| modality | object | False | str_validator |
| location | object | False

In [ ]:
#View the schema of the variant registry
print(REGISTRY_SCHEMAS["variant"].to_markdown_table())

| name | dtype | required | validator |
| --- | --- | --- | --- |
| id | object | True | None |
| label | object | True | None |
| idx | <class 'numpy.int64'> | True | None |
| active | <class 'numpy.bool'> | True | None |
| created_at | object | True | None |
| created_by | object | True | None |
| modified_at | object | True | None |
| modified_by | object | True | None |
| chrom | object | False | str_validator |
| start_pos | i8 | False | None |
| end_pos | i8 | False | None |
| ref_allele | object | False | str_validator |
| alt_allele | object | False | str_validator |
| hugo | object | False | str_validator |
| entrez_gene_id | object | False | str_validator |
| variant_classification | object | False | str_validator |
| variant_type | object | False | str_validator |
| dbsnp_id | object | False | str_validator |
| filter | object | False | str_validator |
| info | object | False | str_validator |
| source | object | False | str_validator |


In [ ]:
#View the schema of the variant registry
print(REGISTRY_SCHEMAS["snp"].to_markdown_table())

| name | dtype | required | validator |
| --- | --- | --- | --- |
| id | object | True | None |
| label | object | True | None |
| idx | <class 'numpy.int64'> | True | None |
| active | <class 'numpy.bool'> | True | None |
| created_at | object | True | None |
| created_by | object | True | None |
| modified_at | object | True | None |
| modified_by | object | True | None |
| chrom | S10 | True | None |
| start_pos | i8 | True | None |
| ref_allele | S10 | True | None |
| alt_allele | S10 | True | None |
| dbsnp_id | S20 | False | None |
| strand | S1 | False | None |


## Add entities to the registry
There are multiple options to register entities. Registry items can be added directly to a `Registry` object via the low-level `Registry.add(...)` function from a list of dictionaries, where each dictionary is an entity.  

But there is also an `io` class to help streamline insertation from standard formats ('maf' files) and CSV and TSV files. 

### Low-level insertion
To utilize low-level insertation, first obtain a referenece to a 'Registry' object and then call the `add(...)` function on a list of dictionaries.

In [24]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    reg.add([
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ])

    print(f"After add, sample registry contains {len(reg)} entities.")

Sample registry contains 0 entities.
After add, sample registry contains 2 entities.


In [25]:
with DNAStream.open(myfile, mode="r+") as ds:
       
    ds.variant.add([
        {"chrom": "chr1", "start_pos": 1231, "end_pos": 1232, "ref_allele": "A", "alt_allele": "T"},
    ])

    for snv in ds.variant:
        print(f"SNV id: {snv['id']}, SNV label: {snv['label']}, active {snv['active']}")

SNV id: 44c7a694-ffe9-4a4e-ad70-fb993b767f04, SNV label: chr1:1231:A:T, active True


There are two key parameters to the `add(...)` function which deal with the fact that internal `labels` are not required to be unique:
- `allow_duplicate_labels` : whether to allow the insertation to proceed with inserting new records that collide with existng registere entities (default is False).
- `activate_newest` : in the event of duplicate labels, this specifies whether to activate the new entity upon insertation (and deactivate the old entity) or whether to leave the newly inserted colliding entity deactivated.

In [26]:
#open stream in mode 'r+' for registry dataset modification.
with DNAStream.open(path=myfile, mode="r+") as ds:

    #pointer to the built-in sample registry
    reg = ds.sample
    print(f"Sample registry contains {len(reg)} entities.")

    reg.add([
        {"sample_name": "S1", "modality": "bulk"},
        {"sample_name": "S2", "modality": "single-cell"},
    ], allow_duplicate_labels=True, activate_newest=True)

    print(f"After add, sample registry contains {len(reg)} entities.")

Sample registry contains 2 entities.
After add, sample registry contains 4 entities.


Now we can iterate through all of the sample entities in the dataset and view the following fields `label`, `id`, `active`, `sample_name`, `created_at`. Note that there are two copies of each label but only newest inserted entity for that label is activated.

In [27]:
with DNAStream.open(path=myfile) as ds:
    for sample in ds.sample:
        print(f'label: {sample["label"]}, id: {sample["id"]}, active: {sample["active"]}, sample_name {sample["sample_name"]}, created at:{sample["created_at"]}')

label: S1, id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50, active: False, sample_name S1, created at:2026-02-05T18:47:20.209014Z
label: S2, id: b9a1464f-8c75-4c31-89ce-b0326805f334, active: False, sample_name S2, created at:2026-02-05T18:47:20.209014Z
label: S1, id: fa35199d-cf8b-4126-8d7a-11e75cb075bc, active: True, sample_name S1, created at:2026-02-05T18:47:20.228717Z
label: S2, id: 6adcdb67-eafe-4e81-9546-84b4cc3900af, active: True, sample_name S2, created at:2026-02-05T18:47:20.228717Z


## Key Python special methods
You may have noticed their use above that the `Registry` interface implements special Python methods for ease of use:
- `__len__`: get the number of registered entities,
- `__iter__` : iterate through each entity in the registry
- `__get__` : slice the registry using the unique id of an entry
- `__contains__` : check if a unique id is present in the registry 

In [28]:
with DNAStream.open(myfile) as ds:
    
    nrecords = len(ds.sample) #get the number of entities
    print(f"The 'sample' registry contains {nrecords}")

    #iterate through the registry yielding each row as a dictionary
    for record in ds.variant:
        print(record)
        my_id = record["id"]
    
    #slice the variant registry dataset by id
    record =ds.variant[my_id]
    print(record)

    # check if my_id is present in the variant registry dataset
    print(f"id: {my_id} present in variant registry: {my_id in ds.variant}") 

    #check if my_id is present in the sample registry dataset
    print(f"id: {my_id} present in sample registry: {my_id in ds.sample}") 


The 'sample' registry contains 4
{'id': '44c7a694-ffe9-4a4e-ad70-fb993b767f04', 'label': 'chr1:1231:A:T', 'idx': 0, 'active': True, 'created_at': '2026-02-05T18:47:20.218432Z', 'created_by': 'llweber', 'modified_at': '2026-02-05T18:47:20.218432Z', 'modified_by': 'llweber', 'chrom': 'chr1', 'start_pos': 1231, 'end_pos': 1232, 'ref_allele': 'A', 'alt_allele': 'T', 'hugo': '', 'entrez_gene_id': '', 'variant_classification': '', 'variant_type': '', 'dbsnp_id': '', 'filter': '', 'info': '', 'source': ''}
{'id': '44c7a694-ffe9-4a4e-ad70-fb993b767f04', 'label': 'chr1:1231:A:T', 'idx': 0, 'active': True, 'created_at': '2026-02-05T18:47:20.218432Z', 'created_by': 'llweber', 'modified_at': '2026-02-05T18:47:20.218432Z', 'modified_by': 'llweber', 'chrom': 'chr1', 'start_pos': 1231, 'end_pos': 1232, 'ref_allele': 'A', 'alt_allele': 'T', 'hugo': '', 'entrez_gene_id': '', 'variant_classification': '', 'variant_type': '', 'dbsnp_id': '', 'filter': '', 'info': '', 'source': ''}
id: 44c7a694-ffe9-4a4e-

## Look up registered entities by `label` or `id`
The entity id is the only unique identifier for a registered entity. As described, labels are used to provide a standardized but non-unique identifier for an entity. Therefore, it is neccessary to convert from a label to and id or from id to a label. There are multiple functions to support this conversion:
- `resolve_ids` : given ids, resolve the 'activated' labels
- `resolved_labels` : given labels, resolve the UUIDv4 id
- `find_ids`: given labels, find all ids assosciated with each label

In [29]:
#look up the unique ids for activated labels
labels = ["S1", "S2"]

with DNAStream.open(myfile) as ds:
    ids =ds.sample.resolve_ids(labels=labels)
    for id, label in zip(ids, labels):
        print(f"id: {id} for label {label}")

id: fa35199d-cf8b-4126-8d7a-11e75cb075bc for label S1
id: 6adcdb67-eafe-4e81-9546-84b4cc3900af for label S2


In [30]:
#look up activated labels given the ids
with DNAStream.open(myfile) as ds:
    labels =ds.sample.resolve_labels(ids=ids)
    for id, label in zip(ids, labels):
        print(f"id: {id} for label {label}")



id: fa35199d-cf8b-4126-8d7a-11e75cb075bc for label S1
id: 6adcdb67-eafe-4e81-9546-84b4cc3900af for label S2


However, since we added our samples to the sample registry twice, there are multiple ids associated with each label.  To look up all ids associated with a label, used `find_ids()`.  This returns a dictionary with the key as the label and the value as a numpy array containing all ids registered for that label.

In [31]:
with DNAStream.open(myfile) as ds:
    all_ids =ds.sample.find_ids(labels=labels)
    for key, found_ids in all_ids.items():
        print(f"label: {key}: {found_ids}")

label: S1: ['cffff0d0-02d2-4f93-8a25-e6ed04a7ee50'
 'fa35199d-cf8b-4126-8d7a-11e75cb075bc']
label: S2: ['b9a1464f-8c75-4c31-89ce-b0326805f334'
 '6adcdb67-eafe-4e81-9546-84b4cc3900af']


## Update the metadata of registered entities
As described above, only certain fields in a registry are modifiable. Registry datasets are only modifiable via the `Registry.update(...)` function.  To provide additional guardrails, the unique id of an entity is needed to modify its metadata.  In this example, we will look up the unique ids associated with sample 'S1' and then we will change its 'modality' in the registry to 'lcm'.  We pass update a list of dictionaries, where each dictionary must key contain the key 'id' and any fieldnames with associated values to update.

In [32]:

with DNAStream.open(myfile, mode="r+") as ds:
    print("Before....")
    for entry in ds.sample:
        print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")
    print("After....")
    sample_ids =ds.sample.find_ids(labels="S1")
    rows = [{"id": id, "modality": "lcm"} for id in sample_ids["S1"]]
    ds.sample.update(rows, warn_missing=True)
    for entry in ds.sample:
        print(f"id: {entry['id']}, label: {entry['label']}, modality: {entry['modality']}")

Before....
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50, label: S1, modality: bulk
id: b9a1464f-8c75-4c31-89ce-b0326805f334, label: S2, modality: single-cell
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc, label: S1, modality: bulk
id: 6adcdb67-eafe-4e81-9546-84b4cc3900af, label: S2, modality: single-cell
After....
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50, label: S1, modality: lcm
id: b9a1464f-8c75-4c31-89ce-b0326805f334, label: S2, modality: single-cell
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc, label: S1, modality: lcm
id: 6adcdb67-eafe-4e81-9546-84b4cc3900af, label: S2, modality: single-cell


# Activate and deactivate registered entities
One of the main advantages of registering entities is that all users can view and track which the current set of research entities to include in their downstream analysis. `Registry` interfaces provides functionality to activate and deactive entities by 'id' or 'label'.  **Note:** Caution should be used when activating and deactivating entities by label as there may be multiple labels. Review the DNAStream policy and pay careful attention to function arguments to ensure proper activation occurs.

Supposed sample 'S1' fails QA, lets deactivate all entities associated with sample 'S1'. 

In [33]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.deactivate_labels(labels="S1")

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: False
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: True
After...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: False
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: False


Now, let's reactivate sample "S1" by label. We will use the default value of `activate_newest` to ensure the most recent sample 'S1' entity added to the registry is activated.

In [34]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")
       
    ds.sample.activate_labels(labels="S1")

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: False
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: False
After...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: False
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: True


Finally, let's suppose that want to activate the other sample 'S1' instead. We can use its unique id to ensure that entity is activated.  Any other entities with the same label will then be deactivated.

In [35]:
with DNAStream.open(myfile, mode="r+") as ds:
    print("Before...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

    id_dict = ds.sample.find_ids(labels="S1")   
    for id in id_dict["S1"]:
        if not ds.sample[id]["active"]:
            my_id = id
    ds.sample.activate_ids(ids = my_id)

    print("After...")
    for row in ds.sample:
        if row["label"] == "S1":
            print(f"id: {row['id']} active: {row['active']}")

Before...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: False
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: True
After...
id: cffff0d0-02d2-4f93-8a25-e6ed04a7ee50 active: True
id: fa35199d-cf8b-4126-8d7a-11e75cb075bc active: False


## Query/export the registry or a subset of the registry to a `pandas.Dataframe`
Users can query or export views of a registry dataset as a `pandas.Dataframe` via two different functions:`to_dataframe(...)` and `get(...)` and . Note, these only return copies of the dataset as a dataframe and any modifications of this dataframe will not be reflected in the DNAStream HDF5 file.  

The `to_dataframe()` function is a convenient way to extract the entire registry, or typical subsets, such as all active entities or only deactivated entities. This is passed through the `mode` parameter.

In [36]:
with DNAStream.open(myfile) as ds:
    sample_registry_df = ds.sample.to_dataframe(mode="all") #default
    print(f"Shape:{sample_registry_df.shape}")
    print(sample_registry_df)

    active_samples_df = ds.sample.to_dataframe(mode="active_only")
    print(f"Shape:{active_samples_df.shape}")

    non_active_samples_df = ds.sample.to_dataframe(mode="non_active")
    print(f"Shape:{non_active_samples_df.shape}")



Shape:(4, 26)
                                     id label  idx  active  \
0  cffff0d0-02d2-4f93-8a25-e6ed04a7ee50    S1    0    True   
1  b9a1464f-8c75-4c31-89ce-b0326805f334    S2    1   False   
2  fa35199d-cf8b-4126-8d7a-11e75cb075bc    S1    2   False   
3  6adcdb67-eafe-4e81-9546-84b4cc3900af    S2    3    True   

                    created_at created_by                  modified_at  \
0  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.275577Z   
1  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.209014Z   
2  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.275577Z   
3  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.228717Z   

  modified_by sample_name organism  ... center_name run study coverage  \
0     llweber          S1           ...                            NaN   
1     llweber          S2           ...                            NaN   
2     llweber          S1           ...                            NaN   
3     ll

However, users may want to make more complex queries to the registry return the records as a `pandas.Dataframe`. For this, the `get(...)` function should be used instead of `to_dataframe()`.  The key argument to `get(...)` is selector. `selector` can be list of labels, ids or a callable function that will subset a `pandas.Dataframe`.

In [37]:
#subset the sample registry by entities with label "S1"
with DNAStream.open(myfile) as ds:

    S1_df = ds.sample.get(selector=["S1"], by="label")
    print(S1_df)

                                     id label  idx  active  \
0  cffff0d0-02d2-4f93-8a25-e6ed04a7ee50    S1    0    True   
1  fa35199d-cf8b-4126-8d7a-11e75cb075bc    S1    2   False   

                    created_at created_by                  modified_at  \
0  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.275577Z   
1  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.275577Z   

  modified_by sample_name organism  ... center_name run study coverage  \
0     llweber          S1           ...                            NaN   
1     llweber          S1           ...                            NaN   

  modality location bam_file_path batch_id reference_build  date_of_sequencing  
0      lcm                                                                      
1      lcm                                                                      

[2 rows x 26 columns]


In [38]:
#subset the sample registry by entities with label "S1"
with DNAStream.open(myfile) as ds:
    all_ids = ds.sample.find_ids("S1")
    
    s1_ids = all_ids["S1"]

    S1_df = ds.sample.get(selector=s1_ids, by="id")
    print(S1_df)

                                     id label  idx  active  \
0  cffff0d0-02d2-4f93-8a25-e6ed04a7ee50    S1    0    True   
1  fa35199d-cf8b-4126-8d7a-11e75cb075bc    S1    2   False   

                    created_at created_by                  modified_at  \
0  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.275577Z   
1  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.275577Z   

  modified_by sample_name organism  ... center_name run study coverage  \
0     llweber          S1           ...                            NaN   
1     llweber          S1           ...                            NaN   

  modality location bam_file_path batch_id reference_build  date_of_sequencing  
0      lcm                                                                      
1      lcm                                                                      

[2 rows x 26 columns]


In [39]:
#subset the sample registry by all entities with 'single-cell' modality
with DNAStream.open(myfile) as ds:
    

    cell_df = ds.sample.get(lambda x: x['modality']=='single-cell')
    print(cell_df)

                                     id label  idx  active  \
1  b9a1464f-8c75-4c31-89ce-b0326805f334    S2    1   False   
3  6adcdb67-eafe-4e81-9546-84b4cc3900af    S2    3    True   

                    created_at created_by                  modified_at  \
1  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.209014Z   
3  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.228717Z   

  modified_by sample_name organism  ... center_name run study coverage  \
1     llweber          S2           ...                            NaN   
3     llweber          S2           ...                            NaN   

      modality location bam_file_path batch_id reference_build  \
1  single-cell                                                   
3  single-cell                                                   

   date_of_sequencing  
1                      
3                      

[2 rows x 26 columns]


## DNAstream I/O functions
Typically, users will want to register a large number of entities from the output files of other upstream methods, like variant callers (maf files) or via CSV and TSV files.  `DNAStream` provides `io` wrappers to streamline the insertation of entities to built-in registries from files. 

Currently, `DNAStream` offers three `io` functions to add facilitate this:
1. `io.add_variants_from_maf`: add variants to the variant registry from a maf file or a list of maf files.
2. `io.add_snps_from_maf` : add SNPSs to the SNP registry from a maf file or a list of maf files.
3. `io.add_samples_from_files` : add samples to the sample registry from a CSV (default) or other delimited file.

Additionally, `DNAStream` offers static functions to parse CSV (`io.parse_csv`, `io.parse_tsv`) or TSV files for preparation in the list of dictionary format required by `Registry.add(...)`. 

First, we will generate some temporary files with example data.

In [40]:
import tempfile

maf_text = "Hugo_Symbol\tChromosome\tStart_Position\tEnd_Position\tReference_Allele\tTumor_Seq_Allele2\tTumor_Sample_Barcode\nTP53\t17\t7579472\t7579472\tC\tT\tS1\n"
csv_text = "sample_name,modality\nS3,bulk\nS4,single-cell\n"

with tempfile.NamedTemporaryFile("w", suffix=".maf", delete=False) as maf_fp:
    maf_fp.write(maf_text)
    maf_path = maf_fp.name

with tempfile.NamedTemporaryFile("w", suffix=".csv", delete=False) as csv_fp:
    csv_fp.write(csv_text)
    csv_path = csv_fp.name

maf_path, csv_path

('/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/tmpoi90kjqp.maf',
 '/var/folders/lp/5zy1x0zj4d514x9hznjdys880000gr/T/tmp8dpcjbww.csv')

In [41]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Sample registry started with {len(ds.sample)} entities.")
    ds.io.add_samples_from_files(str(csv_path))
    print(f"Sample registry now contains {len(ds.sample)} entities.")
    print(ds.sample.to_dataframe())

Sample registry started with 4 entities.
Sample registry now contains 6 entities.
                                     id label  idx  active  \
0  cffff0d0-02d2-4f93-8a25-e6ed04a7ee50    S1    0    True   
1  b9a1464f-8c75-4c31-89ce-b0326805f334    S2    1   False   
2  fa35199d-cf8b-4126-8d7a-11e75cb075bc    S1    2   False   
3  6adcdb67-eafe-4e81-9546-84b4cc3900af    S2    3    True   
4  af52a255-4b26-4cb6-ac08-92463368c554    S3    4    True   
5  05ee383c-0868-4be7-a0e3-4eed719301ed    S4    5    True   

                    created_at created_by                  modified_at  \
0  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.275577Z   
1  2026-02-05T18:47:20.209014Z    llweber  2026-02-05T18:47:20.209014Z   
2  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.275577Z   
3  2026-02-05T18:47:20.228717Z    llweber  2026-02-05T18:47:20.228717Z   
4  2026-02-05T18:47:20.374037Z    llweber  2026-02-05T18:47:20.374037Z   
5  2026-02-05T18:47:20.374037Z    llweb

In [42]:
with DNAStream.open(myfile, mode="r+") as ds:
    print(f"Variant registry started with {len(ds.variant)} entities.")
    ds.io.add_variants_from_maf(str(maf_path))
    print(f"Variant registry now contains {len(ds.variant)} entities.")
    print(ds.variant.to_dataframe())

Variant registry started with 1 entities.
Variant registry now contains 2 entities.
                                     id              label  idx  active  \
0  44c7a694-ffe9-4a4e-ad70-fb993b767f04      chr1:1231:A:T    0    True   
1  72754023-bb64-4813-a5b0-357e4f2b69ec  chr17:7579472:C:T    1    True   

                    created_at created_by                  modified_at  \
0  2026-02-05T18:47:20.218432Z    llweber  2026-02-05T18:47:20.218432Z   
1  2026-02-05T18:47:20.388002Z    llweber  2026-02-05T18:47:20.388002Z   

  modified_by chrom  start_pos  ...  ref_allele alt_allele  hugo  \
0     llweber  chr1       1231  ...           A          T         
1     llweber    17    7579472  ...           C          T  TP53   

  entrez_gene_id variant_classification variant_type dbsnp_id filter info  \
0                                                                           
1                                                                           

  source  
0         
1       